In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
%matplotlib inline

from torch.utils.data import DataLoader, Dataset

In [221]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [222]:
with open("data/output_chat.txt") as f:
    data = f.read()

print(f"Length of dataset in characters: {len(data)}")

Length of dataset in characters: 571276


In [223]:
# Create a set of unique characters in the dataset
chars = sorted(set(data))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")

# Create a mapping from characters to indices and vice versa
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [char_to_idx[c] for c in s]
decode = lambda l: ''.join([idx_to_char[i] for i in l])

n1 = int(len(data) * 0.8)
n2 = int(len(data) * 0.9)
train_data = data[:n1]
val_data = data[n1:n2]
test_data = data[n2:]

print(f"Length of training set: {len(train_data)}")
print(f"Length of validation set: {len(val_data)}")
print(f"Length of test set: {len(test_data)}")

Vocab size: 363
Length of training set: 457020
Length of validation set: 57128
Length of test set: 57128


In [224]:
class CharDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.block_size]
        y = self.data[idx + 1:idx + self.block_size + 1]
        return torch.tensor(encode(x), dtype=torch.long, device=device), \
            torch.tensor(encode(y), dtype=torch.long, device=device)

# Hyperparameter
block_size = 64  # Length of each sequence

train_dataset = CharDataset(train_data, block_size)
val_dataset = CharDataset(val_data, block_size)
test_dataset = CharDataset(test_data, block_size)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

Number of training samples: 456956
Number of validation samples: 57064
Number of test samples: 57064


In [225]:
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Example: Iterate through the DataLoader
for batch_idx, (x, y) in enumerate(train_loader):
    print(f"Batch {batch_idx}:")
    print(f"x: {x}, {x.shape}")
    print(f"y: {y}, {y.shape}")
    break  # Print only the first batch

Batch 0:
x: tensor([[ 72,  16,   2,  47,  65,  73,  78,  69,   2,  80,  79,  79,  67,  72,
          65,   2,  75,  65,  72,  65,  78,   2,  67,  72,  65,  76,  69,   2,
          71,  65,  89,  69,  16,   2,  54,  85,  77,  78,  69,   2,  66,  79,
          76,  65,   2,  83,  72,  79,  80,  80,  73,  78,  71,   2,  75,  65,
          82,   2,  82,  65,  72,  65,   2,  84],
        [ 82,  69,   2,  76,  73,  89,  69,   2,  80,  79,  67,  75,  69,  84,
           2,  83,  81,  85,  65,  82,  69,   2,  79,  82,  68,  69,  82,   2,
          75,  73,  65,   2,  72,  65,  73,  16,   2, 188,   1,  48,  73,  84,
          73,  78,  28,   2,  78,  79,  73,  67,  69,  16,   2,  66,  72,  69,
          74,  79,   2,  75,  65,  85,  78,  83],
        [ 28,   2,  52,  67,  72,  68,   2,  80,  72,  79,  66,  79,  16,   2,
          35,  66,  72,  73,   2,  65,  73,  82,  80,  79,  82,  84,   2,  80,
          69,   2,  72,  73,   2,  72,  65,  73,  78,  16,   2,  40,  76,  73,
          71,  72, 

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim) # One-hot encoding
        self.rnn = nn.LSTM(emb_dim, emb_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(emb_dim, emb_dim * 4),
            nn.ReLU(),
            nn.Linear(emb_dim * 4, vocab_size)
        )

    def forward(self, x):
        emd = self.embedding(x)  # One-hot encoding
        out, _ = self.rnn(emd)  # RNN layer
        out = self.fc(out[:, -1, :])  # Use the last time step's output
        return out


In [ ]:
model = RNN(vocab_size, emb_dim=vocab_size).to(device)
print(model)

print(f"Number of model params: {sum(p.numel() for p in model.parameters())}")

# Example: Iterate through the DataLoader once
for batch_idx, (x, y) in enumerate(train_loader):
    print(f"Batch {batch_idx}:")
    print(f"x: {x}, {x.shape}")
    print(f"y: {y}, {y.shape}")
    out = model(x)
    print(f"Out shape = {out.shape}")
    break

RNN(
  (embedding): Embedding(363, 363)
  (rnn): LSTM(363, 726, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=726, out_features=1452, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1452, out_features=363, bias=True)
  )
)
Number of model params: 4883076
Batch 0:
x: tensor([[  2,  49,  75,  65,  89,  16,   2,  57,  65,  83,  72,  82,  79,  79,
          77,   2,  75,  69,   2,  80,  69,  69,  67,  72,  69,   2,  75,  65,
          80,  68,  69,   2,  67,  72,  69,  67,  75,   2,  75,  65,  82,   2,
          76,  69,  78,  65,  16,   2,  54,  85,  77,  72,  65,  82,  65,   2,
          76,  65,  80,  84,  79,  80,   2,  67],
        [ 78,  71,   2,  75,  73,   2,  76,  79,  67,  65,  84,  73,  79,  78,
           2,  83,  72,  65,  82,  69,   2,  75,  65,  82,  79,   2,  65,  80,
          78,  73,  16,   2,  16,  16,  16,   2,  55,  78,  65,  86,  65,  73,
          76,  65,  66,  76,  69,   2,  65,  65,   2,  82,  65,  72,  65,   2,
          72,  65,  73, 

In [228]:
@torch.no_grad()
def evaluate(model, criterion, split="val"):
    if split == "val":
        loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)
    else:
        loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)
    avg_loss = 0.0
    num_correct = 0
    for batch_idx, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        y = y[:, -1]  # Use the last time step's target
        avg_loss += criterion(logits, y) / len(loader)
        num_correct += (logits.argmax(1) == y).sum().item()
    return avg_loss.item(), num_correct*100 / len(loader.dataset)


In [229]:
import math
from torch.optim.lr_scheduler import LambdaLR

def get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))  # cosine decay
    return LambdaLR(optimizer, lr_lambda)

In [230]:
# Traning loop
epochs = 3
lr = 0.01
batch_size = 1024

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = get_cosine_schedule_with_warmup(optimizer, 100, epochs * len(train_loader))

# Define the loss function
criterion = nn.CrossEntropyLoss()
losses = []
lossesi = []
val_losses = []


for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch_idx, (x, y) in enumerate(train_loader):
        optimizer.zero_grad()
        logits = model(x)
        # print(f"Logits shape = {logits.shape}, {y[:, -1].shape}")
        loss = criterion(logits, y[:, -1])  # Use the last time step's output
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item()}, Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
            losses.append(total_loss / (batch_idx + 1))
            val_loss, accuracy = evaluate(model, criterion, split="val")
            val_losses.append(val_loss)
            lossesi.append(lossesi[-1] + 1 if len(lossesi) > 0 else 0)
    
    val_loss, accuracy = evaluate(model, criterion, split="val")
    print(f"Avg Train Loss: {total_loss/len(train_loader)}, val_loss: {val_loss:.4f}, Accuracy: {accuracy:.2f}%")
    print(f"Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
    print("------")

Epoch 0, Batch 0/447, Loss: 5.902367115020752, Learning Rate: 0.000100
Epoch 0, Batch 100/447, Loss: 2.0313799381256104, Learning Rate: 0.010000
Epoch 0, Batch 200/447, Loss: 1.8408637046813965, Learning Rate: 0.009837
Epoch 0, Batch 300/447, Loss: 1.808470368385315, Learning Rate: 0.009367
Epoch 0, Batch 400/447, Loss: 1.6612191200256348, Learning Rate: 0.008617
Avg Train Loss: 1.9853408384643145, val_loss: 1.7949, Accuracy: 50.21%
Learning Rate: 0.008192
------
Epoch 1, Batch 0/447, Loss: 1.5992697477340698, Learning Rate: 0.008182
Epoch 1, Batch 100/447, Loss: 1.6277905702590942, Learning Rate: 0.007115
Epoch 1, Batch 200/447, Loss: 1.6136400699615479, Learning Rate: 0.005913
Epoch 1, Batch 300/447, Loss: 1.5213476419448853, Learning Rate: 0.004652
Epoch 1, Batch 400/447, Loss: 1.4446218013763428, Learning Rate: 0.003414
Avg Train Loss: 1.550122254913552, val_loss: 1.6311, Accuracy: 55.23%
Learning Rate: 0.002874
------
Epoch 2, Batch 0/447, Loss: 1.4722850322723389, Learning Rate: 

In [ ]:
plt.figure(figsize=(10, 6))

# Plot losses and val_losses
plt.plot(lossesi, losses, label="Training Loss", marker='o')
plt.plot(lossesi, val_losses, label="Validation Loss", marker='x')

# Add trend lines
z_train = np.polyfit(lossesi, losses, 1)
p_train = np.poly1d(z_train)
plt.plot(lossesi, p_train(lossesi), linestyle='--', color='blue', alpha=0.7, label="Training Trend")

z_val = np.polyfit(lossesi, val_losses, 1)
p_val = np.poly1d(z_val)
plt.plot(lossesi, p_val(lossesi), linestyle='--', color='orange', alpha=0.7, label="Validation Trend")

# Add grid lines, labels, and legend
plt.grid(True, linestyle='--', alpha=0.6)
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
import time
def sample(n, context=None):
    model.eval()
    if context is None:
        result = [torch.randint(0, vocab_size, (1,)).item()]
    else:
        result = encode(context[-block_size:])

    with torch.no_grad():
        for _ in range(n):
            x = torch.tensor(result[-block_size:], dtype=torch.long).unsqueeze(0).to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=-1)
            idx = torch.multinomial(probs, num_samples=1)
            result.append(idx.item())
            if len(result) > block_size:
                print(decode(result[:-block_size]), end="")
                result = result[-block_size:]
    return decode(result)

print(sample(2000, "Nitin: "))

Nitin: 
Nitin: Choti si. 4 kha lo. Jigna toh
Nitin: Haha. Choti siiii. Kyunki. tumne thodi laga lo jate hi zyada waise acha tha padi ho tum. Itne cab lana hi hota hai jana kha lo tum. bol rha hai
Nitin: create has rehna yaaaaaaaaaa
Wifey <3: Pisika. Haha. Ek wala phobo. Bagu
Nitin: 👍. Nice the doggo hi booo. kau tumne neverment na nahi. Aur upar hain. Tum to relone ke hi hain. Kahan chalo. ya fufu. Mai guession lur hain. Abhi ghar nahi. Shelly ogarded. Kaha ho. Phobo. Hotely😒. Try
Nitin: Haha
Wifey <3: Jiske lia bolti ho tum. Kaha ho. gaya ho. Main aya mai. Muka aaoge. Khana haare hi tumhara lekin
Nitin: tum complicate ho. Kiss this 😘
Wifey <3: Nahi. And negry nahi nahi hai. Nikal jaunga theek hai. Cuteeee ho wapis
Wifey <3: 😂
Nitin: https://abnb.me/
Wifey <3: Hahaha
Nitin: haan. Doctor hai mil gaya hai mujhe to meeting
Wifey <3: Kyu
Wifey <3: ❤️. Ok bye
Nitin: Haan wo wahi?
Wifey <3: Kyu. Is cover pe to nightes nahi thi. I. Sehneu sapne hate
Wifey <3: Haan. Haan relined ke leke. Itna 

: 